# 均方误差 (MSE) 实现与应用

MSE是机器学习中最常用的损失函数，本notebook演示MSE的计算、梯度、以及在线性回归中的应用。

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 2. 实现MSE和相关指标

In [ ]:
def mse(y_true, y_pred):
    """计算均方误差"""
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    """计算均方根误差"""
    return np.sqrt(mse(y_true, y_pred))

def mae(y_true, y_pred):
    """计算平均绝对误差"""
    return np.mean(np.abs(y_true - y_pred))

def mse_gradient(y_true, y_pred):
    """计算MSE梯度"""
    n = len(y_true)
    return (2.0 / n) * (y_pred - y_true)

## 3. 基础MSE计算示例

In [ ]:
# 示例数据
y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred = np.array([1.1, 2.3, 2.8, 4.2, 4.9])

# 计算各种误差指标
mse_value = mse(y_true, y_pred)
rmse_value = rmse(y_true, y_pred)
mae_value = mae(y_true, y_pred)

print("示例数据：")
print(f"真实值: {y_true}")
print(f"预测值: {y_pred}")
print(f"\n误差指标：")
print(f"MSE:  {mse_value:.6f}")
print(f"RMSE: {rmse_value:.6f}")
print(f"MAE:  {mae_value:.6f}")

# 可视化误差
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
x = np.arange(len(y_true))
plt.bar(x - 0.2, y_true, 0.4, label='真实值', alpha=0.7)
plt.bar(x + 0.2, y_pred, 0.4, label='预测值', alpha=0.7)
plt.xlabel('样本索引')
plt.ylabel('值')
plt.title('真实值 vs 预测值')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
errors = y_pred - y_true
plt.bar(x, errors, alpha=0.7, color='red')
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.xlabel('样本索引')
plt.ylabel('误差')
plt.title('预测误差')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. MSE vs MAE 损失函数对比

In [ ]:
# 对比不同损失函数
y_true_single = 0.0
y_pred_range = np.linspace(-5, 5, 100)

mse_losses = (y_true_single - y_pred_range) ** 2
mae_losses = np.abs(y_true_single - y_pred_range)

plt.figure(figsize=(10, 6))
plt.plot(y_pred_range, mse_losses, 'b-', linewidth=2, label='MSE (平方误差)')
plt.plot(y_pred_range, mae_losses, 'r-', linewidth=2, label='MAE (绝对误差)')
plt.axvline(x=y_true_single, color='g', linestyle='--', alpha=0.5, label='真实值 = 0')
plt.xlabel('预测值')
plt.ylabel('损失值')
plt.title('MSE vs MAE 损失函数比较')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("观察：")
print("- MSE对远离真实值的预测惩罚更重（二次增长）")
print("- MAE对所有误差一视同仁（线性增长）")
print("- MSE在真实值处导数为0，MAE不可导")

## 5. 异常值对MSE的影响

In [ ]:
# 正常数据
y_normal = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred_normal = np.array([1.1, 2.1, 3.1, 4.1, 5.1])

# 添加异常值
y_outlier = np.append(y_normal, 100.0)
y_pred_outlier = np.append(y_pred_normal, 1.0)

# 计算指标
mse_normal = mse(y_normal, y_pred_normal)
mae_normal = mae(y_normal, y_pred_normal)

mse_outlier = mse(y_outlier, y_pred_outlier)
mae_outlier = mae(y_outlier, y_pred_outlier)

print("异常值影响分析：")
print("\n正常数据：")
print(f"  MSE: {mse_normal:.6f}")
print(f"  MAE: {mae_normal:.6f}")

print("\n添加异常值后：")
print(f"  MSE: {mse_outlier:.6f} (增加 {mse_outlier/mse_normal:.1f}倍)")
print(f"  MAE: {mae_outlier:.6f} (增加 {mae_outlier/mae_normal:.1f}倍)")

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 正常数据
ax1.scatter(range(len(y_normal)), y_normal, s=100, label='真实值', alpha=0.7)
ax1.scatter(range(len(y_pred_normal)), y_pred_normal, s=100, label='预测值', alpha=0.7)
ax1.set_xlabel('样本索引')
ax1.set_ylabel('值')
ax1.set_title(f'正常数据 (MSE={mse_normal:.4f})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 带异常值的数据
ax2.scatter(range(len(y_outlier)), y_outlier, s=100, label='真实值', alpha=0.7)
ax2.scatter(range(len(y_pred_outlier)), y_pred_outlier, s=100, label='预测值', alpha=0.7)
ax2.set_xlabel('样本索引')
ax2.set_ylabel('值')
ax2.set_title(f'含异常值数据 (MSE={mse_outlier:.4f})')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 实现线性回归（使用MSE）

In [ ]:
class LinearRegressionMSE:
    """使用MSE的线性回归"""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y, verbose=False):
        """训练模型"""
        n_samples, n_features = X.shape
        
        # 初始化参数
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # 梯度下降
        for i in range(self.n_iterations):
            # 前向传播
            y_pred = np.dot(X, self.weights) + self.bias
            
            # 计算损失
            loss = mse(y, y_pred)
            self.loss_history.append(loss)
            
            # 计算梯度
            dw = (2.0 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (2.0 / n_samples) * np.sum(y_pred - y)
            
            # 更新参数
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            if verbose and (i + 1) % 100 == 0:
                print(f"Iteration {i+1}/{self.n_iterations}, Loss: {loss:.6f}")
        
        return self
    
    def predict(self, X):
        """预测"""
        return np.dot(X, self.weights) + self.bias
    
    def score(self, X, y):
        """计算R²得分"""
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

## 7. 线性回归示例

In [ ]:
# 生成线性数据: y = 2x + 1 + noise
X_train = np.linspace(0, 10, 100).reshape(-1, 1)
y_train = 2 * X_train.flatten() + 1 + np.random.randn(100) * 0.5

# 训练模型
model = LinearRegressionMSE(learning_rate=0.01, n_iterations=1000)
model.fit(X_train, y_train, verbose=True)

# 预测
y_pred = model.predict(X_train)

print(f"\n学习到的参数：")
print(f"  权重 w: {model.weights[0]:.4f} (真实值: 2.0)")
print(f"  偏置 b: {model.bias:.4f} (真实值: 1.0)")

print(f"\n模型性能：")
print(f"  MSE:  {mse(y_train, y_pred):.6f}")
print(f"  RMSE: {rmse(y_train, y_pred):.6f}")
print(f"  R²:   {model.score(X_train, y_train):.6f}")

## 8. 可视化线性回归结果

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 拟合结果
ax1.scatter(X_train, y_train, alpha=0.5, label='训练数据')
ax1.plot(X_train, y_pred, 'r-', linewidth=2, label='拟合直线')
ax1.plot(X_train, 2 * X_train.flatten() + 1, 'g--', linewidth=2, label='真实直线', alpha=0.5)
ax1.set_xlabel('X')
ax1.set_ylabel('y')
ax1.set_title('线性回归拟合结果')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 损失曲线
ax2.plot(model.loss_history, linewidth=2)
ax2.set_xlabel('迭代次数')
ax2.set_ylabel('MSE损失')
ax2.set_title('训练损失曲线')
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

## 9. 可视化MSE损失曲面

对于简单的线性回归，可以可视化参数空间中的MSE损失曲面。

In [ ]:
# 生成简单数据
X_simple = np.array([[1], [2], [3], [4], [5]])
y_simple = np.array([2, 4, 6, 8, 10])  # y = 2x

# 创建参数网格
w_range = np.linspace(0, 4, 50)
b_range = np.linspace(-2, 2, 50)
W, B = np.meshgrid(w_range, b_range)

# 计算每个参数组合的MSE
Z = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        y_pred = W[i, j] * X_simple.flatten() + B[i, j]
        Z[i, j] = mse(y_simple, y_pred)

# 3D可视化
fig = plt.figure(figsize=(14, 5))

# 3D曲面图
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(W, B, Z, cmap='viridis', alpha=0.8)
ax1.set_xlabel('权重 w')
ax1.set_ylabel('偏置 b')
ax1.set_zlabel('MSE损失')
ax1.set_title('MSE损失曲面（3D）')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# 等高线图
ax2 = fig.add_subplot(122)
contour = ax2.contour(W, B, Z, levels=20, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)
ax2.plot(2, 0, 'r*', markersize=15, label='最优点 (w=2, b=0)')
ax2.set_xlabel('权重 w')
ax2.set_ylabel('偏置 b')
ax2.set_title('MSE损失等高线图')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("- MSE损失曲面是凸的（对于线性回归）")
print("- 存在唯一的全局最小值")
print("- 梯度下降可以找到最优解")

## 10. 梯度下降过程可视化

In [ ]:
# 记录梯度下降过程
w_history = []
b_history = []
loss_history = []

w, b = 0.5, -1.5  # 初始值
lr = 0.1
n_iters = 50

for _ in range(n_iters):
    w_history.append(w)
    b_history.append(b)
    
    y_pred = w * X_simple.flatten() + b
    loss = mse(y_simple, y_pred)
    loss_history.append(loss)
    
    # 计算梯度
    dw = (2.0 / len(X_simple)) * np.dot(X_simple.T, (y_pred - y_simple))[0]
    db = (2.0 / len(X_simple)) * np.sum(y_pred - y_simple)
    
    # 更新参数
    w -= lr * dw
    b -= lr * db

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 参数更新轨迹
ax1 = axes[0]
ax1.plot(w_history, linewidth=2, label='权重 w')
ax1.plot(b_history, linewidth=2, label='偏置 b')
ax1.axhline(y=2.0, color='r', linestyle='--', alpha=0.5, label='真实 w=2')
ax1.axhline(y=0.0, color='g', linestyle='--', alpha=0.5, label='真实 b=0')
ax1.set_xlabel('迭代次数')
ax1.set_ylabel('参数值')
ax1.set_title('参数更新过程')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 损失下降曲线
ax2 = axes[1]
ax2.plot(loss_history, linewidth=2)
ax2.set_xlabel('迭代次数')
ax2.set_ylabel('MSE损失')
ax2.set_title('损失下降曲线')
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# 参数空间中的轨迹
ax3 = axes[2]
contour = ax3.contour(W, B, Z, levels=20, cmap='viridis', alpha=0.5)
ax3.plot(w_history, b_history, 'r.-', linewidth=2, markersize=8, label='梯度下降路径')
ax3.plot(w_history[0], b_history[0], 'go', markersize=12, label='起点')
ax3.plot(w_history[-1], b_history[-1], 'r*', markersize=15, label='终点')
ax3.set_xlabel('权重 w')
ax3.set_ylabel('偏置 b')
ax3.set_title('参数空间中的优化轨迹')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"最终参数: w={w:.4f}, b={b:.4f}")
print(f"真实参数: w=2.0, b=0.0")

## 11. 多项式回归示例

In [ ]:
# 生成非线性数据
X_poly = np.linspace(0, 10, 100).reshape(-1, 1)
y_poly = 0.5 * X_poly.flatten()**2 - 3 * X_poly.flatten() + 5 + np.random.randn(100) * 2

# 创建多项式特征
def polynomial_features(X, degree):
    """生成多项式特征"""
    X_poly = np.zeros((X.shape[0], degree))
    for i in range(degree):
        X_poly[:, i] = X.flatten() ** (i + 1)
    return X_poly

# 测试不同度数
degrees = [1, 2, 3, 5]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, degree in enumerate(degrees):
    ax = axes[idx]
    
    # 生成多项式特征并训练
    X_poly_train = polynomial_features(X_poly, degree)
    model_poly = LinearRegressionMSE(learning_rate=0.001, n_iterations=2000)
    model_poly.fit(X_poly_train, y_poly)
    
    # 预测
    y_pred_poly = model_poly.predict(X_poly_train)
    
    # 计算指标
    mse_val = mse(y_poly, y_pred_poly)
    r2_val = model_poly.score(X_poly_train, y_poly)
    
    # 绘图
    ax.scatter(X_poly, y_poly, alpha=0.5, label='数据')
    ax.plot(X_poly, y_pred_poly, 'r-', linewidth=2, label='拟合曲线')
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.set_title(f'度数={degree}\nMSE={mse_val:.2f}, R²={r2_val:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. 不同学习率的影响

In [ ]:
# 测试不同学习率
learning_rates = [0.001, 0.01, 0.1, 0.5]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, lr in enumerate(learning_rates):
    ax = axes[idx]
    
    # 训练模型
    model_lr = LinearRegressionMSE(learning_rate=lr, n_iterations=100)
    model_lr.fit(X_train, y_train)
    
    # 绘制损失曲线
    ax.plot(model_lr.loss_history, linewidth=2)
    ax.set_xlabel('迭代次数')
    ax.set_ylabel('MSE损失')
    ax.set_title(f'学习率 = {lr}')
    ax.grid(True, alpha=0.3)
    
    if max(model_lr.loss_history) > 100:
        ax.set_yscale('log')

plt.tight_layout()
plt.show()

print("观察：")
print("- 学习率太小：收敛慢")
print("- 学习率太大：可能发散或震荡")
print("- 合适的学习率：快速且稳定地收敛")

## 13. 总结

通过本notebook，我们深入探索了MSE损失函数：

1. **基础计算**：实现了MSE、RMSE、MAE等误差指标
2. **损失函数对比**：MSE对大误差惩罚更重，对异常值更敏感
3. **线性回归**：使用MSE训练线性回归模型
4. **损失曲面**：可视化参数空间中的MSE损失曲面
5. **梯度下降**：展示了参数优化的过程
6. **多项式回归**：扩展到非线性拟合

**关键要点**：
- MSE是回归任务的标准损失函数
- MSE具有良好的数学性质（凸、可导、唯一解）
- MSE对异常值敏感，需要根据数据特点选择
- 梯度下降可以有效优化MSE
- 学习率的选择对收敛速度和稳定性很重要